In [1]:
from tqdm.auto import tqdm
import pandas as pd
import polars as pl
import numpy as np
import argparse
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from typing import Optional
from typing import Tuple
from typing import List
from typing import Dict
from typing import Any

In [2]:
DATA_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
SPEC_DIR = os.path.join(DATA_DIR, "test_spectrograms/")
EEG_DIR = os.path.join(DATA_DIR, "test_eegs/")

df_train = pl.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test = pl.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

df_test.head()

spectrogram_id,eeg_id,patient_id
i64,i64,i64
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
8352559,1706196508,37552


In [3]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cpu



In [4]:
def load_model(path: str, model: nn.Module) -> nn.Module:
    model.load_state_dict(torch.load(path, map_location = torch.device("cpu"))["model_state_dict"])
    return model

In [5]:
import sys
sys.path.append("/kaggle/input/hms-models/")

from eeg_cnn_rnn_knn import EegModel

CurrModel = EegModel

ModuleNotFoundError: No module named 'eeg_cnn_rnn_knn'

In [6]:
ensembles = []

fold_dirs = [
    #"/kaggle/input/hms-models/eeg_cnn_rnn_stage_3/*",
    #"/kaggle/input/hms-models/simple_eeg_cnn_rnn_stage_2/*",
    #"/kaggle/input/hms-models/softmax_simple_eeg_cnn_rnn_stage_2/softmax_simple_eeg_cnn_rnn_stage_2/*"
    #"/kaggle/input/hms-models/baseline_eeg_diff/*"
    "/kaggle/input/hms-models/knn_0/*"
]

import glob
import sys

models = [] 
for fold_dir in fold_dirs:
    for fold_path in glob.glob(fold_dir):
        model_path = os.path.join(fold_path, "model_best_val_g10.pt")
        model = CurrModel()
        model.from_pretrained(fold_path)
        model = model.to(device)
        models.append(model)
        
len(models)

0

In [7]:
import polars as pl
import librosa
import numpy as np

from scipy.ndimage import convolve

KERNEL = np.array([-1, -1, -1, 0, 1, 1, 1])

def compute_spec(eeg: np.ndarray) -> np.ndarray:
    spectrogram = librosa.stft(
        eeg, n_fft = 1024, hop_length = 39, win_length = 256, 
        window = "hann", center = True, pad_mode = "constant", 
        out = None, dtype = None
    )
    spectrogram = librosa.power_to_db(
        np.abs(spectrogram) ** 2, ref = np.max
    ).astype(np.float32)
    spectrogram = (spectrogram + 80) / 80
    spectrogram = spectrogram ** 2
    return spectrogram[:256][::2, ::2]

def spec(eeg: np.ndarray) -> np.ndarray:
    spec = compute_spec(eeg)
    eeg = convolve(eeg, KERNEL)
    spec = spec + compute_spec(eeg)
    
    return spec / 2

def compute_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    F3  = df_eeg["F3"].to_numpy()
    F4  = df_eeg["F4"].to_numpy()
    F7  = df_eeg["F7"].to_numpy()
    F8  = df_eeg["F8"].to_numpy()
    C3  = df_eeg["C3"].to_numpy()
    C4  = df_eeg["C4"].to_numpy()
    P3  = df_eeg["P3"].to_numpy()
    P4  = df_eeg["P4"].to_numpy()
    T3  = df_eeg["T3"].to_numpy()
    T4  = df_eeg["T4"].to_numpy()
    T5  = df_eeg["T5"].to_numpy()
    T6  = df_eeg["T6"].to_numpy()
    O1  = df_eeg["O1"].to_numpy()
    O2  = df_eeg["O2"].to_numpy()
    
    ll = np.stack([(spec(Fp1 - F7), spec(F7 - T3), spec(T3 - T5), spec(T5 - O1))])
    lp = np.stack([(spec(Fp1 - F3), spec(F3 - C3), spec(C3 - P3), spec(P3 - O1))])
    rp = np.stack([(spec(Fp2 - F4), spec(F4 - C4), spec(C4 - P4), spec(P4 - O2))])
    rl = np.stack([(spec(Fp2 - F8), spec(F8 - T4), spec(T4 - T6), spec(T6 - O2))])
    
    chain = np.stack([ll, lp, rp, rl])[:, 0]
    
    return chain

def compute_spec_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_chain(df_eeg)
    chain = preprocess_chain(chain)
    return chain

In [8]:
import numpy as np
import math

from typing import Union
from typing import Tuple
from typing import List


def bin_array(
    array, 
    bin_size, 
    axis = -1, 
    pad_dir = "symmetric", 
    mode = "edge", 
    return_padding = False,
    **padding_kwargs
) -> Union[np.ndarray, Tuple[np.ndarray, List]]:
    """Given an array and bin size, bins the array along an arbitrary axis into
    bins of size `bin_size`. It will perform padding if the array does not split
    up into equal bin sizes. 

    Args:
        array (np.ndarray): The input array.
        bin_size (int): The size of each bin.
        axis (int): The axis to bin the array along. Default is -1.
        pad_dir (str): The padding direction. One of `left`, `right`, or
            `symmetric` (default).
        return_padding (bool): Option to return the padding width used. 
        mode (str): The padding mode. See the NumPy documentation for options. 

    Returns:
        np.ndarray: The binned array where the number of bins is first. That is,
            the shape will be `(..., n_bins, bin_size, ...)`.

    """
    if axis == -1:
        axis = array.ndim - 1

    curr_len = array.shape[axis]
    n_bins = math.ceil(curr_len / bin_size)
    new_len = n_bins * bin_size

    new_shape = list(array.shape)
    new_shape[axis] = n_bins
    new_shape.insert(axis + 1, bin_size)

    # Perform padding if curr_len does not equal new_len.
    padding = [(0, 0)] * array.ndim
    if curr_len != new_len:
        if pad_dir == "left":
            pad_l = new_len - curr_len
            pad_r = 0
        elif pad_dir == "right":
            pad_l = 0
            pad_r = new_len - curr_len
        else:
            pad_l = (new_len - curr_len) // 2
            pad_r = (new_len - curr_len) - pad_l

        padding[axis] = (pad_l, pad_r)
        array = np.pad(array, padding, mode = mode, **padding_kwargs)

    array = array.reshape(new_shape)

    if return_padding:
        return array, padding

    return array

In [9]:
from scipy.signal import welch
from scipy.stats import linregress
from scipy.signal import butter, filtfilt

def butter_filter(eeg_data, fs = 200, cutoff_freq = 22, order = 4, btype = "lowpass"):
    b, a = butter(
        N = order, 
        Wn = cutoff_freq / (0.5 * fs), 
        btype = btype, 
        analog = False
    )

    return filtfilt(b, a, eeg_data)

def compute_eeg(eeg: np.ndarray) -> np.ndarray:
    eeg = butter_filter(eeg, cutoff_freq = np.array([0.25, 50]), btype = "bandpass")
    eeg = bin_array(eeg, bin_size = 4, mode = "reflect").mean(axis = -1)
    return eeg
    
def compute_eeg_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    F3  = df_eeg["F3"].to_numpy()
    F4  = df_eeg["F4"].to_numpy()
    F7  = df_eeg["F7"].to_numpy()
    F8  = df_eeg["F8"].to_numpy()
    C3  = df_eeg["C3"].to_numpy()
    C4  = df_eeg["C4"].to_numpy()
    P3  = df_eeg["P3"].to_numpy()
    P4  = df_eeg["P4"].to_numpy()
    T3  = df_eeg["T3"].to_numpy()
    T4  = df_eeg["T4"].to_numpy()
    T5  = df_eeg["T5"].to_numpy()
    T6  = df_eeg["T6"].to_numpy()
    O1  = df_eeg["O1"].to_numpy()
    O2  = df_eeg["O2"].to_numpy()
    
    ll = np.stack([(compute_eeg(Fp1 - F7), compute_eeg(F7 - T3), compute_eeg(T3 - T5), compute_eeg(T5 - O1))])
    lp = np.stack([(compute_eeg(Fp1 - F3), compute_eeg(F3 - C3), compute_eeg(C3 - P3), compute_eeg(P3 - O1))])
    rp = np.stack([(compute_eeg(Fp2 - F4), compute_eeg(F4 - C4), compute_eeg(C4 - P4), compute_eeg(P4 - O2))])
    rl = np.stack([(compute_eeg(Fp2 - F8), compute_eeg(F8 - T4), compute_eeg(T4 - T6), compute_eeg(T6 - O2))])
    
    chain = np.stack([ll, lp, rp, rl])[:, 0]
    
    return chain

def compute_eeg_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_eeg_chain(df_eeg)
    
    return chain

In [10]:
# Option for better printing.
np.set_printoptions(formatter = {"all": lambda x: f"{x:0.3f}"})

@torch.no_grad()
def gen_ensemble_pred(models, df_row: pl.DataFrame) -> np.ndarray:
    eeg_id = df_row["eeg_id"].item()
    filepath = os.path.join(EEG_DIR, f"{eeg_id}.parquet")
    
    x = compute_eeg_from_file(filepath)
    x[np.isnan(x) | np.isinf(x)] = 0
    x = torch.Tensor(x).to(device)
    x = x - x.mean(dim = -1, keepdim = True)
    x = x / (x.std(dim = -1, keepdim = True) + 1e-5)
    x = x.reshape(16, 1, 2_500)
    x = x.unsqueeze(0)

    preds = []
    for model in models:
        model.eval()
        #pred = model(x).exp()
        pred = model.knn_predict(x)
        pred = pred.cpu().detach().numpy().reshape(-1)
        preds.append(pred)
        
    for pred in preds:
        print(pred)
        
    preds = np.mean(preds, axis = 0)
    preds = preds / preds.sum()
    return preds

In [11]:
LABELS = [
    "seizure_vote", 
    "lpd_vote", 
    "gpd_vote", 
    "lrda_vote", 
    "grda_vote", 
    "other_vote"
]

In [12]:
from tqdm.auto import tqdm
    
preds_final = []
for i in range(len(df_test)):
    pred = gen_ensemble_pred(models, df_test[i])
    #x = load_eeg(df_test[i])
    #pred = model.predict(x)
    preds_final.append(pred)

RuntimeError: shape '[16, 1, 2500]' is invalid for input of size 65600

In [13]:
df_sub = pd.DataFrame({"eeg_id": df_test["eeg_id"].to_list()})
df_sub[LABELS] = preds_final
df_sub.to_csv("submission.csv", index = False)
df_sub

ValueError: Columns must be same length as key